# 第 9 章实验：卷积核、stride 与边界
在合成图上观察手工卷积核响应。修改“参数”那一格后从头运行，不需要任何图片文件。

第一格会打印当前用的设备（有 NVIDIA 显卡并且装了带 CUDA 的 PyTorch 时是 `cuda`，否则是 `cpu`）。输出尺寸那一格和最后一格会调用你在 `作业_卷积shape与残差块.py` 里写的函数，作业还没完成时只打印提示。

In [ ]:
import importlib.util
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 图里的中文：Windows 自带的字体放在前面，找不到再试 macOS 和 Linux 上的
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Heiti SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# 设备：有 NVIDIA 显卡就用 CUDA，没有就用 CPU。本实验很小，在 CPU 上也只要几秒到几十秒
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| 设备:", device, "|",
      torch.cuda.get_device_name(0) if device.type == "cuda" else "没有检测到 CUDA，用 CPU 运行")

CHAPTER_NAME = "09_CNN核心"
HOMEWORK_FILE = "作业_卷积shape与残差块.py"


def load_homework():
    """载入本章的作业文件。设置了环境变量 COURSE_IMPL_DIR 时改从那个目录载入（维护课程时用）。"""
    here = Path.cwd().resolve()
    chapter = None
    for p in [here, *here.parents]:
        if p.name == CHAPTER_NAME:
            chapter = p
            break
        if (p / CHAPTER_NAME).is_dir():
            chapter = p / CHAPTER_NAME
            break
    if chapter is None:
        raise FileNotFoundError("请在课程目录内启动 Jupyter")
    base = chapter
    override = os.environ.get("COURSE_IMPL_DIR")
    if override:
        root = Path(override)
        if not root.is_absolute() and not (root / CHAPTER_NAME).is_dir():
            root = chapter.parent / root   # 相对路径按课程根目录算
        base = root / CHAPTER_NAME
    spec = importlib.util.spec_from_file_location("homework", base / HOMEWORK_FILE)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def attempt(fn, *args, **kwargs):
    """调用作业里的函数；还没实现时打印提示并返回 None，不中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {getattr(fn, '__name__', type(fn).__name__)}：{exc}")
        return None


hw = load_homework()
print("作业文件：", HOMEWORK_FILE)

In [ ]:
# 参数：改这里，然后从头运行
size = 32      # 图像边长，可改为 31
stride = 1
padding = 1
kernel_name = 'vertical'  # 可改为 horizontal
torch.manual_seed(9)

In [ ]:
image = torch.zeros(1, 1, size, size)
image[:, :, 7:25, 9:23] = 1.0
image[:, :, torch.arange(5, 28), torch.arange(3, 26)] = 0.5
vertical = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
horizontal = vertical.T.contiguous()
kernel = {'vertical': vertical, 'horizontal': horizontal}[kernel_name].reshape(1, 1, 3, 3)
image, kernel = image.to(device), kernel.to(device)   # 输入和卷积核要在同一个设备上
response = F.conv2d(image, kernel, stride=stride, padding=padding)
print('input NCHW:', tuple(image.shape), 'output:', tuple(response.shape))
print('先用教程 9.2 节的公式手算输出边长，再和上面的 output 对照。')
predicted = attempt(hw.conv_output_size, size, 3, stride, padding)
if predicted is not None:
    print('你的 conv_output_size 给出的边长:', predicted)
    assert response.shape[-2:] == (predicted, predicted), '作业函数的结果和 PyTorch 的实际输出不一致'

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(image[0, 0].cpu(), cmap='gray', vmin=0, vmax=1); axes[0].set_title('合成输入')
axes[1].imshow(kernel[0, 0].cpu(), cmap='coolwarm'); axes[1].set_title(kernel_name + ' 卷积核')
axes[2].imshow(response[0, 0].detach().cpu(), cmap='coolwarm'); axes[2].set_title(f'响应 {tuple(response.shape[-2:])}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 破坏实验：padding 改变边界假设
比较零填充与反射填充。反射填充不会在边缘突然接上黑色，但也不一定适合所有任务。

上面那张合成图的内容都不贴边，最外一圈全是 0，两种填充算出来完全一样。所以这一格另外画一个贴着左上角的方块：差别只出现在图像最外一圈，里面一个像素都不变。

In [ ]:
def both_paddings(img):
    return F.conv2d(img, kernel, padding=1), F.conv2d(F.pad(img, (1, 1, 1, 1), mode='reflect'), kernel)

zero_padded, reflected = both_paddings(image)
print('内容不贴边的原图：两种填充的最大差', float((zero_padded - reflected).abs().max()))
edge_image = torch.zeros_like(image)
edge_image[:, :, 0:18, 0:14] = 1.0   # 贴着左上角的方块
zero_padded, reflected = both_paddings(edge_image)
edge_difference = (zero_padded - reflected).abs()
ring = torch.ones_like(edge_difference, dtype=torch.bool)
ring[..., 1:-1, 1:-1] = False   # 最外一圈
print('内容贴边的方块：最外一圈的差异均值', float(edge_difference[ring].mean()), '| 里面的最大差', float(edge_difference[..., 1:-1, 1:-1].max()))
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(zero_padded[0, 0].cpu(), cmap='coolwarm'); axes[0].set_title('零填充')
axes[1].imshow(reflected[0, 0].cpu(), cmap='coolwarm'); axes[1].set_title('反射填充')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
layers = [(3, 1), (2, 2), (3, 1)]
r, jump = 1, 1
for kernel_size, layer_stride in layers:
    r = r + (kernel_size - 1) * jump
    jump *= layer_stride
    print(f'kernel={kernel_size}, stride={layer_stride} -> receptive_field={r}, jump={jump}')

## 接上你的作业：残差块
用你在 `作业_卷积shape与残差块.py` 里写的 `BasicResidualBlock` 跑一次前向，再做教程 9.10 节的检查：把第二个 BN 的缩放系数置 0，推理模式下残差分支的输出是 0，整个块的输出应该和 `relu(x)` 完全相同。作业还没完成时这一格只打印提示。

In [ ]:
torch.manual_seed(9)
block = hw.BasicResidualBlock(4).to(device).eval()   # 先在 CPU 上初始化，再搬到 device
x = torch.randn(2, 4, 16, 16).to(device)
with torch.no_grad():
    y = attempt(block, x)
if y is not None:
    print('输入', tuple(x.shape), '输出', tuple(y.shape), '| 输出里负数的个数:', int((y < 0).sum()))
    nn.init.zeros_(block.bn2.weight)
    with torch.no_grad():
        difference = float((block(x) - torch.relu(x)).abs().max())
    print('bn2 的缩放系数置 0 之后，输出和 relu(x) 的最大差:', difference, '（应为 0）')